In [ ]:
import pandas as pd
import spacy
import re

nlp = spacy.load("en_core_web_sm")

df = pd.read_csv("dataset.csv")
BAD_ORGS = {
    "Reuters",
    "REUTERS",
    "Reuters Staff",
    "CNBC",
    "BBC",
    "Fortune",
    "PRNewswire",
    "Globe Newswire",
    "Eikon",
    "SEDAR",
}

def clean_company_name(name):
    name = str(name).strip()
    name = re.sub(r"\s+", " ", name)
    name = name.strip(".,;:()[]{}\"'")
    return name

def is_probably_company(name):
    if not name:
        return False

    if name in BAD_ORGS:
        return False

    if len(name) < 2:
        return False

    generic_terms = {
        "Reporting",
        "Editing",
        "Source",
        "Conference Call",
        "Forward-Looking Statements",
        "Non-GAAP Financial Measures",
    }

    if name in generic_terms:
        return False

    return True


def extract_companies(text):
    if pd.isna(text):
        return []

    doc = nlp(str(text))

    companies = []

    for ent in doc.ents:
        if ent.label_ == "ORG":
            company = clean_company_name(ent.text)

            if is_probably_company(company):
                companies.append(company)

    # Deduplicate while preserving order
    seen = set()
    unique_companies = []

    for company in companies:
        key = company.lower()

        if key not in seen:
            seen.add(key)
            unique_companies.append(company)

    return unique_companies


rows = []

for article_id, row in df.iterrows():
    companies = extract_companies(row["text"])
    if article_id % 100 == 0:
        print(f"Processing article {article_id} / {len(df)}")
    for company in companies:
        rows.append({
            "article_id": article_id,
            "date": row["date"],
            "title": row["title"],
            "company": company,
            "url": row["url"]
        })

company_df = pd.DataFrame(rows)
company_df.groupby(["article_id", "date", "title", "url"], as_index=False).agg({"company": list})
company_df.to_csv("companies_in_articles.csv", index=False)

print(company_df.head(20))
print(f"Saved {len(company_df)} article-company rows.")

Processing article 0 / 1000
Processing article 100 / 1000
Processing article 200 / 1000
Processing article 300 / 1000
Processing article 400 / 1000
Processing article 500 / 1000
Processing article 600 / 1000
Processing article 700 / 1000
Processing article 800 / 1000
Processing article 900 / 1000
    article_id        date                                              title  \
0            0  2018-04-26  McLaren review F1 technical operations, Goss m...   
1            0  2018-04-26  McLaren review F1 technical operations, Goss m...   
2            0  2018-04-26  McLaren review F1 technical operations, Goss m...   
3            0  2018-04-26  McLaren review F1 technical operations, Goss m...   
4            1  2018-05-15  ADDvantage Technologies Announces Financial Re...   
5            1  2018-05-15  ADDvantage Technologies Announces Financial Re...   
6            1  2018-05-15  ADDvantage Technologies Announces Financial Re...   
7            1  2018-05-15  ADDvantage Technologies An